In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.7
)

In [17]:
class JokeState(TypedDict):
    topic: str
    joke:str
    explanation:str
    

In [18]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the given topic: {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke':response}

In [19]:
def generate_explanation(state: JokeState):
    prompt = f'Provide me the detailed explaination of the given joke {state["joke"]}'
    response = llm.invoke(prompt).content 

    return {'explanation': response } 

In [20]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [22]:
config1 = {"configurable": {"thread_id": "1"}}

workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough!**',
 'explanation': 'This joke is a classic example of a **pun**, which is a form of wordplay that exploits multiple meanings of a word, or of words that sound alike but have different meanings.\n\nHere\'s the detailed breakdown:\n\n1.  **The Setup:** "Why did the pizza get a job?"\n    *   This sets up an expectation for a reason why someone (or something personified) would seek employment. The most common reason for a person to get a job is to earn money.\n\n2.  **The Punchline:** "Because it **kneaded dough!**"\n\n3.  **The Double Meaning (The Pun):**\n\n    *   **Meaning 1 (Literal, Pizza-Related):**\n        *   **"Kneaded" (pronounced /niːdɪd/):** This is the past tense of the verb "to knead." To knead means to work (dough or clay) with the hands, pressing and stretching it. This is an essential step in making pizza, bread, and other baked goods.\n        *   **"Dough":** This refers to the m

In [23]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough!**', 'explanation': 'This joke is a classic example of a **pun**, which is a form of wordplay that exploits multiple meanings of a word, or of words that sound alike but have different meanings.\n\nHere\'s the detailed breakdown:\n\n1.  **The Setup:** "Why did the pizza get a job?"\n    *   This sets up an expectation for a reason why someone (or something personified) would seek employment. The most common reason for a person to get a job is to earn money.\n\n2.  **The Punchline:** "Because it **kneaded dough!**"\n\n3.  **The Double Meaning (The Pun):**\n\n    *   **Meaning 1 (Literal, Pizza-Related):**\n        *   **"Kneaded" (pronounced /niːdɪd/):** This is the past tense of the verb "to knead." To knead means to work (dough or clay) with the hands, pressing and stretching it. This is an essential step in making pizza, bread, and other baked goods.\n        *   **"Dough":** T

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough!**', 'explanation': 'This joke is a classic example of a **pun**, which is a form of wordplay that exploits multiple meanings of a word, or of words that sound alike but have different meanings.\n\nHere\'s the detailed breakdown:\n\n1.  **The Setup:** "Why did the pizza get a job?"\n    *   This sets up an expectation for a reason why someone (or something personified) would seek employment. The most common reason for a person to get a job is to earn money.\n\n2.  **The Punchline:** "Because it **kneaded dough!**"\n\n3.  **The Double Meaning (The Pun):**\n\n    *   **Meaning 1 (Literal, Pizza-Related):**\n        *   **"Kneaded" (pronounced /niːdɪd/):** This is the past tense of the verb "to knead." To knead means to work (dough or clay) with the hands, pressing and stretching it. This is an essential step in making pizza, bread, and other baked goods.\n        *   **"Dough":** 